在 Python 中，函数名加上 _ 前缀（如
  _get_obs、_get_info）是一种命名约定，表示该方法是"私有的"（private）。
                                                                                     
  作用：
                                                                                     
  1. 表明这是内部实现细节 — 不建议外部代码直接调用
  2. 封装性 — 隐藏不对外公开的方法，只暴露公共接口（如 reset(), step(), render() 等）
  3. 避免与子类方法冲突 — 明确标识哪些方法是供内部使用的

  常见约定：
  
| 写法           | 本质   | 是否真的私有 | 主要作用          |
| ------------ | ---- | ------ | ------------- |
| `_method`    | 约定   | ❌ 否    | 提示“内部使用”      |
| `__method`   | 名称重整 | ⚠️ 半私有 | 防止子类覆盖        |
| `__method__` | 语言协议 | ❌ 否    | Python 内置特殊方法 |

 
self 代表当前实例对象本身，让方法能够访问该对象的具体属性和数据。                  
                  
  class Agent:                                                                       
      def __init__(self):
          self._agent_location = (0, 0)  # 实例属性

      def _get_obs(self):  # 必须传 self
          return {"agent": self._agent_location}  # 访问自己的属性

  原因：

  1. 访问实例属性 — 方法需要通过 self.属性名 来读取/修改该对象的特有数据
  2. 区分实例方法 vs 类方法/静态方法 — 没有 self 的是类方法或静态方法
  3. 在方法内部调用其他方法 — self._get_info() 调用同类其他方法

In [113]:
from typing import Optional
import numpy as np
import gymnasium as gym 

class GridWorldEnv(gym.Env):
    metadata = {"render_modes": ["human", "rgb_array"]}

    
    def __init__(self, size: int = 5, render_mode = None):
        self.size = size
        self.render_mode = render_mode
        
        self._agent_location = np.array([-1,-1], dtype=np.int32)
        self._target_location = np.array([-1,-1],dtype=np.int32)
        
        # Define what the agent can observe
        # Dict space gives us structured, human-readable observations
        self.observation_space = gym.spaces.Dict(
            {
                "agent": gym.spaces.Box(0, size - 1, shape=(2,), dtype=int),
                "target": gym.spaces.Box(0, size-1, shape=(2,),dtype=int) # [x,y]
            }
        )
        
        self.action_space = gym.spaces.Discrete(4)
        
        self._action_to_direction = {
            0: np.array([0, 1]),   # Move right (column + 1)
            1: np.array([-1, 0]),  # Move up (row - 1)
            2: np.array([0, -1]),  # Move left (column - 1)
            3: np.array([1, 0]),   # Move down (row + 1)
        }
        
    def _get_obs(self):
        """Convert internal state to observation format.

        Returns:
            dict: Observation with agent and target positions
        """
        return {"agent": self._agent_location, "target": self._target_location}
    
    def _get_info(self):
        # return info with dist
        return {
            "distance": np.linalg.norm(
                self._agent_location - self._target_location,ord=1 # manhattan distance
            )
        }
        
    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):
        """Start a new episode.

        Args:
            seed: Random seed for reproducible episodes
            options: Additional configuration (unused in this example)

        Returns:
            tuple: (observation, info) for the initial state
        """
        # IMPORTANT: Must call this first to seed the random number generator
        super().reset(seed=seed)

        # Randomly place the agent anywhere on the grid
        self._agent_location = self.np_random.integers(0, self.size, size=2, dtype=int)

        # Randomly place target, ensuring it's different from agent position
        self._target_location = self._agent_location
        while np.array_equal(self._target_location, self._agent_location):
            self._target_location = self.np_random.integers(
                0, self.size, size=2, dtype=int
            )

        observation = self._get_obs()
        info = self._get_info()

        return observation, info
    
    def step(self, action):
        direction = self._action_to_direction[action]
            # np.clip prevents the agent from walking off the edge
        self._agent_location = np.clip(
            self._agent_location + direction, 0, self.size - 1
        )
        # Check if agent reached the target
        terminated = np.array_equal(self._agent_location, self._target_location)

        # We don't use truncation in this simple environment
        # (could add a step limit here if desired)
        truncated = False
        
        # Simple reward structure: +1 for reaching target, 0 otherwise
        # Alternative: could give small negative rewards for each step to encourage efficiency
        # 0 makes learning ver difficult
        # reward = 1 if terminated else 0 
        
        # Op 1: small step penalty
        reward = 1 if terminated else -0.01
        # Op 2: Distance-based reward shaping
        # distance = np.linalg.norm(self._agent_location - self._target_location)
        # reward = 1 if terminated else -0.1 * distance
        
        observation = self._get_obs()
        info = self._get_info()
        return observation, reward, terminated, truncated, info

    def render(self):
        """Render the environment for human viewing."""
        if self.render_mode == "human":
            # Print a simple ASCII representation
            for y in range(self.size - 1, -1, -1):  # Top to bottom
                row = ""
                for x in range(self.size):
                    if np.array_equal([x, y], self._agent_location):
                        row += "A "  # Agent
                    elif np.array_equal([x, y], self._target_location):
                        row += "T "  # Target
                    else:
                        row += ". "  # Empty
                print(row)  
            print()

In [114]:
# Register the environment so we can create it with gym.make()
gym.register(
    id="gymnasium_env/GridWorld-v0",
    entry_point=GridWorldEnv,
    max_episode_steps=300,  # Prevent infinite episodes
)

/Users/antik/anaconda3/envs/gym_env/lib/python3.11/site-packages/gymnasium/envs/registration.py:636: UserWarning: WARN: Overriding environment gymnasium_env/GridWorld-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")


In [115]:
import gymnasium as gym
env = gym.make("gymnasium_env/GridWorld-v0")
env

<TimeLimit<OrderEnforcing<PassiveEnvChecker<GridWorldEnv<gymnasium_env/GridWorld-v0>>>>>

In [116]:
env = gym.make("gymnasium_env/GridWorld-v0",size=10)
env.unwrapped.size

10

In [117]:
# Create multiple environments for parallel training
vec_env = gym.make_vec("gymnasium_env/GridWorld-v0", num_envs=3)
vec_env

SyncVectorEnv(gymnasium_env/GridWorld-v0, num_envs=3)

In [118]:
from gymnasium.utils.env_checker import check_env

# This will catch many common issues
try:
    check_env(env)
    print("Environment passes all checks!")
except Exception as e:
    print(f"Environment has issues: {e}")
    

. . . . . . . A . . 
. . . . . . . . . . 
. T . . . . . . . . 
. . . . . . . . . . 
. . . . . . . . . . 
. . . . . . . . . . 
. . . . . . . . . . 
. . . . . . . . . . 
. . . . . . . . . . 
. . . . . . . . . . 

Environment passes all checks!


/Users/antik/anaconda3/envs/gym_env/lib/python3.11/site-packages/gymnasium/utils/env_checker.py:384: UserWarning: WARN: The environment (<TimeLimit<OrderEnforcing<PassiveEnvChecker<GridWorldEnv<gymnasium_env/GridWorld-v0>>>>>) is different from the unwrapped version (<GridWorldEnv<gymnasium_env/GridWorld-v0>>). This could effect the environment checker as the environment most likely has a wrapper applied to it. We recommend using the raw environment for `check_env` using `env.unwrapped`.
  logger.warn(
/Users/antik/anaconda3/envs/gym_env/lib/python3.11/site-packages/gymnasium/utils/passive_env_checker.py:334: UserWarning: WARN: No render fps was declared in the environment (env.metadata['render_fps'] is None or not defined), rendering may occur at inconsistent fps.
  logger.warn(
/Users/antik/anaconda3/envs/gym_env/lib/python3.11/site-packages/gymnasium/utils/passive_env_checker.py:270: UserWarning: WARN: RGB-array rendering should return a numpy array, got <class 'NoneType'>
  logger.

In [119]:
env = gym.make("gymnasium_env/GridWorld-v0",render_mode="human")
obs, info = env.reset(seed=42)

print(f"Starting position - Agent: {obs['agent']}, Target: {obs['target']}")
env.render()

actions = [0,1,2,3]
for action in actions:
    old_pos = obs['agent'].copy()
    obs, reward, terminated, truncated, info = env.step(action)
    new_pos = obs['agent']
    print(f"Action {action}: {old_pos} -> {new_pos}, reward={reward}")
    env.render()

Starting position - Agent: [0 3], Target: [3 2]
. . . . . 
A . . . . 
. . . T . 
. . . . . 
. . . . . 

Action 0: [0 3] -> [0 4], reward=-0.01
A . . . . 
. . . . . 
. . . T . 
. . . . . 
. . . . . 

Action 1: [0 4] -> [0 4], reward=-0.01
A . . . . 
. . . . . 
. . . T . 
. . . . . 
. . . . . 

Action 2: [0 4] -> [0 3], reward=-0.01
. . . . . 
A . . . . 
. . . T . 
. . . . . 
. . . . . 

Action 3: [0 3] -> [1 3], reward=-0.01
. . . . . 
. A . . . 
. . . T . 
. . . . . 
. . . . . 



⏺ 是的，FlattenObservation 的作用就是将 observation 展平（flatten）。                                                                          
                                                                                                                                               
  作用                                                                                                                                         
   
  把你的 Dict 或 Tuple 观察空间转换为扁平的 Box：                                                                                              
  ```python              
  import gymnasium as gym
  from gymnasium.wrappers import FlattenObservation

  env = gym.make("gymnasium_env/GridWorld-v0")

  # 原始观察空间
  print(env.observation_space)
  # Dict(Box(0, 4, shape=(2,)), Box(0, 4, shape=(2,)))

  # 包装后
  wrapped_env = FlattenObservation(env)
  print(wrapped_env.observation_space)
  # Box(0, 4, shape=(4,))  # 展平成一维

  效果

  # 原始 obs（字典）
  obs = {"agent": [1, 2], "target": [3, 4]}
  # → {"agent": array([1, 2]), "target": array([3, 4])}

  # 展平后
  obs = wrapped_env.reset()[0]
  print(obs)  # array([1, 2, 3, 4])  # 4个元素的一维数组
```
  用途

  - 方便神经网络处理（很多 NN 需要一维输入）
  - 简化观察空间

In [120]:
def render(self):
    """Render the environment for human viewing."""
    if self.render_mode == "human":
        # Print a simple ASCII representation
        for y in range(self.size - 1, -1, -1):  # Top to bottom
            row = ""
            for x in range(self.size):
                if np.array_equal([x, y], self._agent_location):
                    row += "A "  # Agent
                elif np.array_equal([x, y], self._target_location):
                    row += "T "  # Target
                else:
                    row += ". "  # Empty
            print(row)
        print()

Gymnasium 的规则

Gymnasium 强制规定：

✅ 如果你传了 render_mode

那你必须在环境里声明：

metadata = {"render_modes": ["human", "rgb_array"]}


否则它认为你的环境不支持渲染。

🔥 你现在的情况

你可能写的是：

class GridWorldEnv(gym.Env):
    def __init__(self, render_mode=None):
        self.render_mode = render_mode


但 没有 metadata

于是 Gymnasium 检查到：

render_mode = "human"

但 metadata 里没有 render_modes

直接断言失败。